# Synthetic Data Audit (MVP)
This notebook runs a lightweight audit of `dataset/synth_data.csv` using a TF-IDF + logistic regression pipeline and `cleanlab` to detect potential label errors. Each section has a short description followed by the code cell.

In [19]:
# Imports and setup
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
import numpy as np
from cleanlab.filter import find_label_issues

## 1) Load data and quick overview
Load `dataset/synth_data.csv` and display basic counts and label balance.

In [20]:
# Load dataset
df = pd.read_csv('dataset/synth_data_2000.csv')
print('Rows:', len(df))
print('\nLabel distribution:')
print(df['label'].value_counts())
# Ensure columns exist
assert 'sentence' in df.columns and 'label' in df.columns, 'Expected columns: sentence, label'
texts = df['sentence'].astype(str).values
labels = df['label'].values

Rows: 2000

Label distribution:
label
0    1000
1    1000
Name: count, dtype: int64


## 2) Feature extraction (TF-IDF)
Convert text to TF-IDF features for a simple baseline model.

In [21]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(texts)

## 3) Model training with cross-validated predicted probabilities
Train a logistic regression using 5-fold CV and collect predicted probabilities for `cleanlab`. 

In [22]:
clf = LogisticRegression(max_iter=1000)
pred_probs = cross_val_predict(clf, X, labels, cv=5, method='predict_proba')
print('Predicted probabilities shape:', pred_probs.shape)

Predicted probabilities shape: (2000, 2)


## 4) Detect potential label issues with CleanLab
Use `cleanlab.filter.find_label_issues` to rank examples most likely to be mislabeled.

In [23]:
issue_indices = find_label_issues(labels=labels, pred_probs=pred_probs, return_indices_ranked_by='self_confidence')
print('Found', len(issue_indices), 'potential label issues')
# Preview top 5
for i, idx in enumerate(issue_indices[:5]):
    print(f"\nIssue #{i+1} (Dataset index: {idx})")
    print('Sentence:', df.loc[idx, 'sentence'])
    print('Given label:', df.loc[idx, 'label'])
    print('Predicted probs:', np.round(pred_probs[idx], 3))

Found 0 potential label issues


## 5) Export top suspects for manual review
Save the top 100 ranked examples to a CSV for team validation.

In [24]:
# Export only the indices flagged by cleanlab (no 100 enforcement)
top_indices = list(issue_indices)
if len(top_indices) == 0:
    print('Cleanlab did not flag any issues.')
else:
    suspicious_df = df.iloc[top_indices].copy()
    suspicious_df.insert(0, 'Cleanlab_Rank', range(1, len(suspicious_df) + 1))
    suspicion_scores = pred_probs[top_indices, labels[top_indices]]
    suspicious_df['Suspicion_Score'] = suspicion_scores
    # Leave these blank for manual review
    suspicious_df['Team_Corrected_Label'] = ''
    suspicious_df['Error_Type'] = ''
    suspicious_df['Notes'] = ''
    suspicious_df.to_csv('synth_cleanlab_suspects.csv', index=False)
    print(f'Exported synth_cleanlab_suspects.csv with {len(suspicious_df)} rows')

Cleanlab did not flag any issues.
